In [1]:
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForMaskedLM

/home/sbasir/Thesis/myenv3.10/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
from pymilvus import connections, list_collections

# Connect to Milvus server (adjust the host and port accordingly)
connections.connect(alias="default", host="localhost", port="19530")

# List all existing collections
col = list_collections()
print("Existing collections:", col)


# Load the collection
col = Collection("sbert_experiment")
# Get collection statistics
conn = col._get_connection()
stats = conn.get_collection_stats(col.name)
print(stats[0])
print(type(stats[0]))

# Extract the row_count from stats
row_count = None
for stat in stats:
    if stat.key == "row_count":
        row_count = int(stat.value)  # Convert the string value to an integer
        break
print("Row count:", row_count)

Existing collections: ['sbert_experiment_test', 'sbert_experiment', 'hybrid_experiment', 'hybrid_experiment2']
key: "row_count"
value: "4867173"

<class 'common_pb2.KeyValuePair'>
Row count: 4867173


In [3]:
# read file as list
with open("queries_sample.txt", "r") as f:
    queries_sample = f.readlines()

In [4]:
col.load()

In [5]:
compaction_id = col.compact()


In [6]:
status = col.get_compaction_state(compaction_id)
print(status)



CompactionState
 - compaction id: 452747711439201264
 - State: Executing
 - executing plan number: 10
 - timeout plan number: 0
 - complete plan number: 0



In [13]:
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2', device='cuda')

In [14]:
query = queries_sample[0]
print(query)
query_string = query
query_embeddings = model.encode(query_string)
k=1000

search_params = {"metric_type": "IP"}

# Search topK docs based on dense and sparse vectors and rerank with RRF.
results = col.search(
    data = [query_embeddings],
    anns_field="dense_vector",
    param=search_params,
    limit=10,
    output_fields=["text"]
)

for res in results:
    print(res)

still life musical

["id: /2059218/data_sounds_IT_DDS0000049931000100_6, distance: 0.3829515874385834, entity: {'text': 'l.]: MUSIC.'}", "id: /2048024/Athena_Plus_ProvidedCHO_Biblioteca_nazionale_centrale_di_Roma_oai_bncrm_librari_beniculturali_it_spartito_BVE0348886_1_3, distance: 0.3611319661140442, entity: {'text': '| dc:title is Se la vita. 11 | dc:type is Musical score | dc:date is [1820-1830]/ | dc:creator is Rossini, Gioachino | dc:date is [1820-1830] / | dc:title is If life. 11 | dc:type is Musical score.'}", "id: /0940411/_nnnZk9k_5, distance: 0.3505164682865143, entity: {'text': 'Był wielkim smakoszem życia, wina i jedzenia (pod koniec życia kompozytora dotknął paraliż a potem głuchota). Kompozytor sięgnął po wszystkie główne formy instrumentalnej muzyki baroku. Jego twórczość to opery, pasje, oratoria, serenady, utwory religijne, kantaty, muzyka kameralna i orkiestrowa oraz muzyka na instrumenty klawiszowe.'}", "id: /0940411/_nnnr1Rr_5, distance: 0.3505164682865143, entity: 

In [7]:
import time
from tqdm import tqdm 
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2', device='cuda')

def query_collection(query):
    start = time.time()
    query_string = query
    query_embeddings = model.encode(query_string)
    k=1000

    search_params = {"metric_type": "IP"}

    # Search topK docs based on dense and sparse vectors and rerank with RRF.
    res = col.search(
        data = [query_embeddings],
        anns_field="dense_vector",
        param=search_params,
        limit=k,
        output_fields=["text"]
    )
    del query_embeddings
    end = time.time()
    total_time = end - start
    return total_time

times = []
for query in tqdm(queries_sample):
    total_time = query_collection(query)
    times.append(total_time)

/home/sbasir/Thesis/myenv3.10/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
100%|██████████| 100/100 [01:40<00:00,  1.01s/it]


In [11]:
# save time to file
with open(f"sbert_time_{row_count}_v2.txt", "w") as f:
    for time in times:
        f.write(str(time) + "\n")

2.090738534927368
1.0593187808990479
0.8482506275177002
0.955272912979126
0.9221830368041992
1.0955989360809326
1.0798635482788086
0.8730900287628174
0.9065945148468018
0.9341838359832764
1.012646198272705
1.0151758193969727
0.9825820922851562
1.0021953582763672
0.8994426727294922
1.008570909500122
0.9434230327606201
0.7236626148223877
0.817908763885498
0.7825145721435547
0.8982205390930176
1.021409511566162
0.8625638484954834
0.9501581192016602
0.8884446620941162
0.8646097183227539
0.8641293048858643
0.8665890693664551
0.8470499515533447
1.0531885623931885
1.0692548751831055
0.9121050834655762
1.0491938591003418
1.0297119617462158
1.0936872959136963
1.2416949272155762
0.8998551368713379
0.8554043769836426
0.8137831687927246
0.9151217937469482
0.9419500827789307
0.9487793445587158
1.0908896923065186
0.857813835144043
0.8917577266693115
0.9321072101593018
1.12003755569458
0.9856464862823486
0.8472409248352051
0.7124550342559814
0.6191253662109375
0.899824857711792
0.8836977481842041
0.9